# Taiwan Food 34 — CNN Transfer Learning

**Goal:** 34-way classification of Taiwanese food photos using ImageNet-pretrained CNNs.

**Pipeline (Run all from top):**
1. Check GPU (should be A100 on Colab Pro)
2. Get project code (GitHub clone or upload zip)
3. Install dependencies
4. Upload dataset zip
5. Train ConvNeXt-Tiny (main model)
6. Train ResNet-50 (baseline)
7. Compare results + inline plots
8. Download all outputs as zip

**Expected total runtime on A100:** ~25–35 minutes.

## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Get the project code

**Option A — Clone from your GitHub repo (recommended once you've pushed):**
Edit `GITHUB_URL` below.

**Option B — Upload project zip manually:** if your repo isn't on GitHub yet, change `USE_GITHUB = False` and you'll be prompted to upload a zip of the project folder (containing `src/`, `scripts/`, `configs/`).

In [ ]:
import os, shutil

USE_GITHUB = True
GITHUB_URL = "https://github.com/110207402/taiwan-food-cnn.git"
PROJECT_DIR = "/content/taiwan-food-cnn"

if USE_GITHUB:
    !rm -rf {PROJECT_DIR}
    !git clone {GITHUB_URL} {PROJECT_DIR}
else:
    from google.colab import files
    print("Upload project zip (containing src/, scripts/, configs/, requirements.txt):")
    up = files.upload()
    zip_name = list(up.keys())[0]
    !rm -rf {PROJECT_DIR}
    !mkdir -p {PROJECT_DIR}
    !unzip -q "{zip_name}" -d {PROJECT_DIR}
    junk = os.path.join(PROJECT_DIR, '__MACOSX')
    if os.path.isdir(junk):
        shutil.rmtree(junk)
    if not os.path.isdir(os.path.join(PROJECT_DIR, 'src')):
        children = [c for c in os.listdir(PROJECT_DIR)
                    if os.path.isdir(os.path.join(PROJECT_DIR, c))
                    and not c.startswith('.') and c != '__MACOSX']
        if len(children) == 1:
            inner = os.path.join(PROJECT_DIR, children[0])
            for item in os.listdir(inner):
                shutil.move(os.path.join(inner, item), os.path.join(PROJECT_DIR, item))
            os.rmdir(inner)

os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())
!ls -la

## 2. Install dependencies

In [ ]:
# Re-anchor cwd in case the kernel was restarted between cells.
import os
os.chdir('/content/taiwan-food-cnn')
!pip install -q -r requirements.txt

## 3. 取得資料

**預設用 Google Drive**(快,1GB 大概 3–10 分鐘搞定一次,之後 session 都能重用)。

把 `taiwan_food34.zip` 拖到 Google Drive 根目錄,然後執行下方 cell。

如果你不想用 Drive,把 `DATA_SOURCE = "upload"`,改用 `files.upload()`(慢,大檔案不建議)。

In [ ]:
import os, shutil
os.chdir('/content/taiwan-food-cnn')

# 'drive' = 從 Google Drive 抓(推薦) | 'upload' = 從本機上傳(慢,大檔案不要用)
DATA_SOURCE = "drive"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/taiwan_food34.zip"  # 改成你 Drive 上實際路徑

DATA_DIR = "/content/taiwan-food-cnn/data"
!rm -rf {DATA_DIR} && mkdir -p {DATA_DIR}

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.exists(DRIVE_ZIP_PATH), f"找不到 {DRIVE_ZIP_PATH},請確認 zip 已上傳到 Drive"
    print(f"Copying {DRIVE_ZIP_PATH} ...")
    !cp "{DRIVE_ZIP_PATH}" /content/_data.zip
    zip_path = "/content/_data.zip"
elif DATA_SOURCE == "upload":
    from google.colab import files
    print("Upload taiwan_food34.zip:")
    up = files.upload()
    zip_path = "/content/" + list(up.keys())[0]
else:
    raise ValueError("DATA_SOURCE must be 'drive' or 'upload'")

print(f"Unzipping {zip_path} ...")
!unzip -q "{zip_path}" -d {DATA_DIR}
if zip_path == "/content/_data.zip":
    os.remove(zip_path)

# Clean macOS junk + flatten single-folder wrap
junk = os.path.join(DATA_DIR, '__MACOSX')
if os.path.isdir(junk):
    shutil.rmtree(junk)
if not os.path.isdir(os.path.join(DATA_DIR, 'train')):
    kids = [c for c in os.listdir(DATA_DIR)
            if os.path.isdir(os.path.join(DATA_DIR, c))
            and not c.startswith('.') and c != '__MACOSX']
    if len(kids) == 1:
        inner = os.path.join(DATA_DIR, kids[0])
        for it in os.listdir(inner):
            shutil.move(os.path.join(inner, it), os.path.join(DATA_DIR, it))
        os.rmdir(inner)

# Sanity check
for s in ['train', 'val', 'test']:
    p = os.path.join(DATA_DIR, s)
    if os.path.isdir(p):
        cls = sorted([c for c in os.listdir(p) if os.path.isdir(os.path.join(p, c))])
        n = sum(len(os.listdir(os.path.join(p, c))) for c in cls)
        print(f"  {s}: {len(cls)} classes, {n} images")
    else:
        print(f"  [warn] {s}/ not found at {p}")

## 4. Train ConvNeXt-Tiny (main model)

Trains 3 epochs head-only + up to 25 epochs full fine-tuning with early stopping. Auto-runs test evaluation + error analysis at the end.

In [ ]:
import os
os.chdir('/content/taiwan-food-cnn')
!python -m scripts.run_train --config configs/convnext_tiny.yaml

## 5. Train ResNet-50 (baseline)

In [ ]:
import os
os.chdir('/content/taiwan-food-cnn')
!python -m scripts.run_train --config configs/resnet50.yaml

## 6. Compare models

In [ ]:
import os
os.chdir('/content/taiwan-food-cnn')
!python -m scripts.compare_models --runs outputs/convnext_tiny outputs/resnet50 --out outputs/comparison.csv

In [ ]:
import pandas as pd
pd.read_csv('outputs/comparison.csv')

## 7. View plots inline

In [ ]:
from IPython.display import Image, display, Markdown

for model in ['convnext_tiny', 'resnet50']:
    display(Markdown(f'### {model}'))
    display(Markdown('**Training curves**'))
    display(Image(f'outputs/{model}/train_curves.png'))
    display(Markdown('**Confusion matrix (row-normalized)**'))
    display(Image(f'outputs/{model}/confusion_matrix.png'))
    display(Markdown('**Worst-class misclassifications**'))
    display(Image(f'outputs/{model}/misclassified_samples.png'))

In [ ]:
# Per-class metrics for the main model (sorted worst → best by F1)
import pandas as pd
df = pd.read_csv('outputs/convnext_tiny/per_class_metrics.csv')
df.head(10)

In [ ]:
# Top confused class pairs
pd.read_csv('outputs/convnext_tiny/top_confused_pairs.csv')

## 8. Download all outputs

In [ ]:
!zip -r outputs.zip outputs/ -x 'outputs/**/best.pth'
from google.colab import files
files.download('outputs.zip')

In [ ]:
# Optional: also download model checkpoints separately (large)
# from google.colab import files
# files.download('outputs/convnext_tiny/best.pth')
# files.download('outputs/resnet50/best.pth')